# The Heritage Short — ACE-Step 1.5 Score Generator (3 Candidates)
Generates 3 instrumental Ethiopian score candidates from `storyboard.json` metadata (`caption`, `tempo_bpm`, `key`, `duration_s`) and saves incrementally to `/kaggle/working/outputs/music/`.

In [ ]:
!git clone -q https://github.com/ACE-Step/ACE-Step-1.5.git /kaggle/working/ACE-Step-1.5 || true
!pip install -q -e /kaggle/working/ACE-Step-1.5 soundfile scipy loguru vector-quantize-pytorch
import json, shutil, sys, time
from pathlib import Path
sys.path.insert(0, '/kaggle/working/ACE-Step-1.5')

OUT_DIR = Path('/kaggle/working/outputs/music')
OUT_DIR.mkdir(parents=True, exist_ok=True)

sb_matches = list(Path('/kaggle/input').rglob('storyboard.json'))
if sb_matches:
    sb = json.loads(sb_matches[0].read_text(encoding='utf-8'))
    cue = sb['shots'][0]['music_cue']
else:
    cue = {
        'caption': 'Traditional Ethiopian acoustic score featuring warm plucked krar, airy washint bamboo flute, gentle masinko bowing, and soft kebero heartbeat, reflective tizita pentatonic mood, instrumental only',
        'tempo_bpm': 76,
        'key': 'C major pentatonic',
        'duration_s': 45.0,
    }
print('Caption:', cue['caption'])
print('Metadata:', cue['tempo_bpm'], 'BPM |', cue['key'])

In [ ]:
import os
from acestep.handler import AceStepHandler
from acestep.llm_inference import LLMHandler
from acestep.inference import GenerationParams, GenerationConfig, generate_music

project_root = '/kaggle/working/ACE-Step-1.5'
checkpoint_dir = os.path.join(project_root, 'checkpoints')

dit_handler = AceStepHandler()
dit_handler.initialize_service(project_root=project_root, config_path='acestep-v15-turbo', device='cuda')

llm_handler = LLMHandler()
llm_handler.initialize(checkpoint_dir=checkpoint_dir, lm_model_path='acestep-5Hz-lm-0.6B', backend='pt', device='cuda')

metrics = []
for idx, seed in enumerate((5001, 5002, 5003), start=1):
    final_wav = OUT_DIR / f'candidate_{idx}.wav'
    if final_wav.exists() and final_wav.stat().st_size > 10000:
        print(f'[SKIP] {final_wav}')
        continue
    t0 = time.time()
    params = GenerationParams(
        caption=cue['caption'],
        lyrics='[Instrumental]',
        bpm=int(cue['tempo_bpm']),
        keyscale=cue['key'],
        duration=45,
        seed=seed,
        thinking=True,
    )
    cfg = GenerationConfig(batch_size=1, audio_format='wav', use_random_seed=False, seeds=[seed])
    res = generate_music(dit_handler=dit_handler, llm_handler=llm_handler, params=params, config=cfg, save_dir=str(OUT_DIR))
    elapsed_min = round((time.time() - t0) / 60.0, 3)
    if res.success and res.audios:
        src = Path(res.audios[0]['path'])
        if src.exists() and src != final_wav:
            shutil.copy2(src, final_wav)
        metrics.append({'candidate': idx, 'seed': seed, 'gpu_minutes': elapsed_min, 'path': str(final_wav)})
        Path('/kaggle/working/acestep_metrics.json').write_text(json.dumps(metrics, indent=2))
        print(f'[SAVED] {final_wav} in {elapsed_min:.2f} GPU min')
    else:
        print(f'[FAILED] candidate {idx}: {getattr(res, "error", res)}')